# Training 3 different RNNs

In this notebook I will train 3 different RNNs, one simple RNN, one LSTM and one GRU.

In [1]:
import pandas as pd
import numpy as np
import os
import augumentations as aug
import data_functions as dfunc
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

folder_path = "../../MainProject/data/mediapipe_not_trimmed_world"
score_path = "../../MainProject/data/video_scores.csv"

random_state = 42

## Data Creation

We begin by creating the data we need as well as augumenting it by mirroring and rotating the node network.

In [ ]:
scores = dfunc.load_video_score(score_path=score_path)
files = list(scores["file"])
labels = torch.tensor(list(scores["scaled_score"]))

train_files, test_files, train_y, test_y = train_test_split(files, labels, test_size=0.2, random_state=random_state)

# First and last rotate nothing
rotations = np.linspace(0, 2*np.pi, 5)[:-1]

# Extend the labels to add target cariables for the augumented data
factor = 2 * len(rotations)
train_y = dfunc.extend_tensor(train_y, factor)

def augument_data(files: list[str], suffix: str, rotations: list[float]) -> torch.Tensor:
    """
    Mirrors the data once around x-axis and rotates each mirroring with the angles in the rotation parameter around the z-axis.
    Also standardizes the data.

    Args:
        files: files to augument and load
        suffix: the standard suffix of each file
        rotations: list of angles in radians to rotate each node structure around the z-axis
    Returns:
        Tensor: tensor containing the augumented and normal data of each file in files
    """
    samples = []
    for index, file in enumerate(files):
        path = os.path.join(folder_path, f"{file}{suffix}")

        # Select frames from the file
        df = pd.read_csv(path).drop(columns=["FrameNo"])
        df_sliced = dfunc.select_equally_spaced_rows(df)

        # Augument the data
        # Mirror once
        for mirroring in [True, False]:
            mirrored_data = aug.mirror(df_sliced, mirror_x=mirroring)

            # Rotate for each mirroring
            for angle in rotations:
                rotated_data = aug.rotate(mirrored_data, angle, axis=0)

                # Convert to tensors
                sample = dfunc.create_tensor_from_dataframe(rotated_data)

                # Standardize the data
                scaler = StandardScaler()
                sample = torch.tensor(scaler.fit_transform(sample))

                samples.append(sample)

    return torch.stack(samples)


x_train = augument_data(train_files, "_mediapipe.csv", rotations=rotations)

print(f"Number of training samples: {x_train.shape[0]}")
print(x_train.shape)

Number of training samples: 1120
torch.Size([1120, 30, 39])
